# Notebook 06 — SMAP: MAML-AE against baselines

*Corrected version of the original `06_smap_maml_clean.ipynb` / `06-smap-maml-clean (1).ipynb`.
It also replaces the original `05-evaluation (4).ipynb`, which evaluated the checkpoint from
the broken training loop and whose saved run crashed.*

**What this notebook does.** For each training seed it meta-trains MAML, trains a static
LSTM autoencoder and an MLP autoencoder on the same meta-training channels, and scores all
methods on the seven held-out SMAP channels at 1, 5 and 10 shots. It then summarises the
results across training seeds with confidence intervals and paired differences.

**Why.** This is the experiment behind the manuscript's SMAP table. The original had flaws
that made its numbers hard to interpret (listed below).

**Input.** The raw SMAP release and the `Corrected-Notebooks` repository folder (for `maml_common.py`).
Both are found automatically under `/kaggle/input`.

**Output (in `/kaggle/working`).** One `smap_corrected_seed<seed>.json` per training
seed, `smap_corrected_summary.json`, and model checkpoints.

### How to run on Kaggle

1. Attach two Datasets: the SMAP release, and the `Corrected-Notebooks` folder of this
   repository. Turn on a GPU.
2. Use **Save Version -> Save & Run All (Commit)**.
3. The run is split by training seed. If a session times out, attach this notebook's
   previous output as an extra input and commit again: finished seeds are skipped, and an
   unfinished MAML run resumes from its last checkpoint.
4. Expected time on a Kaggle GPU: about 2.5 hours per training seed (up to 30,000 outer
   steps, based on the original run), so about 12 to 13 hours for five seeds. That exceeds
   one 12-hour session, so plan for one resume. To split the work, set `TRAIN_SEEDS` in
   the configuration cell.

### What was corrected

1. **Point-wise scoring on the test file is the main result.** The original compared
   normal windows from the *training* file with anomaly windows from the *test* file. An
   untrained random network scores 0.542 macro ROC-AUC under that protocol, higher than
   the trained models (see the audit in the README). Now every timestep of each test file
   is scored (stride-1 windows; a timestep's score is the mean error of the windows that
   cover it) against per-timestep labels.
2. **Support and query no longer overlap.** In the original, 12–48% of the normal query
   windows shared timesteps with the support windows the model had just adapted on.
3. **Adaptation steps match training.** MAML was meta-trained with 10 inner steps but
   evaluated after 5. Evaluation now uses 10 steps for every adapted model.
4. **Zero-step controls.** MAML and Static are also scored with no adaptation at all, to
   show whether adaptation changes anything.
5. **"MLP-AE" is now a trained MLP.** The original "MLP-AE" was a random MLP given 50 SGD
   steps on the K support windows, a scratch model. That model is kept under the name
   "MLP-scratch (legacy)"; the new "MLP-AE" is trained like the static LSTM-AE.
6. **An untrained random LSTM-AE is scored** as a floor that any trained model should beat.
7. **F1 is no longer set to 0** when every window is flagged; the anomaly fraction and the
   F1 of "flag everything" are reported next to every F1, and the label-free F1 is kept
   separate from the oracle best-F1.
8. **Isolation Forest at 1 shot is reported as not applicable** (one training point),
   instead of an AUROC of 0.500.
9. **Five independent training seeds** (the original trained once; its "seeds" were only
   support draws). Support draws use separate seeds and are identical across methods and
   training seeds, so comparisons are paired.
10. **Validation uses fixed episodes** and its own random stream.
11. Every result file records the configuration, seeds, code version and time.

**Kept as in the original, so that only the corrections change:** the channel split,
scaling and windows; FOMAML with inner SGD 0.01 x 10 steps, Adam 0.001, 4 tasks per
batch, 20/20 support/query, gradient clipping 1.0, up to 30,000 outer steps, validation
every 500 steps with the learning-rate halving schedule, best-validation checkpoint; the
static recipe (random 90/10 split, Adam 0.001, batch 128, up to 150 epochs, patience 15).

A **legacy view** re-scores every model with the original window protocol and 5
adaptation steps, only so that the new models can be compared with the old table. It is
not a valid test (see point 1).

In [ ]:
import os, sys

def _find_common():
    """Find maml_common.py: this folder when run locally, /kaggle/input on Kaggle."""
    for root in [os.getcwd(), "/kaggle/input"]:
        if os.path.isdir(root):
            for d, _, files in os.walk(root):
                if "maml_common.py" in files:
                    return d
    raise FileNotFoundError("maml_common.py not found. Run from the Corrected-Notebooks folder, "
                            "or attach that folder to Kaggle as a Dataset.")

sys.path.insert(0, _find_common())
import maml_common as sc
SMOKE = os.environ.get("SMAP_SMOKE") == "1"     # tiny settings for testing only
OUT = sc.output_dir(smoke=SMOKE)
print("shared code:", sc.__file__)
print("outputs go to:", OUT)
print("code version:", sc.git_commit())

import copy, json, time, numpy as np, torch
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, roc_auc_score
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| SMOKE TEST (numbers are meaningless)" if SMOKE else "")

## 1 — Configuration

In [ ]:
CFG = dict(
    train_seeds=[42, 123, 456, 789, 1024], support_seeds=[0, 1, 2, 3, 4],
    legacy_support_seeds=[42, 123, 456, 789, 1024], k_shots=[1, 5, 10],
    channels=sc.EVAL_CHANNELS + sc.SENSITIVITY_CHANNELS,
    n_outer=30000, val_every=500, inner_lr=0.01, inner_steps=10, outer_lr=1e-3, tasks_per_batch=4,
    support_size=20, query_size=20, use_scheduler=True, val_episodes_per_task=4, val_seed=2024,
    adapt_steps=10, adapt_lr=0.01, legacy_adapt_steps=5, mlp_scratch_steps=50,
    static_max_epochs=150, static_patience=15,
)
if SMOKE:
    CFG.update(train_seeds=[42], support_seeds=[0], legacy_support_seeds=[42], k_shots=[1, 5],
               channels=["A-6", "D-6"], n_outer=4, val_every=2, val_episodes_per_task=1,
               static_max_epochs=1, static_patience=1)
TRAIN_SEEDS = CFG["train_seeds"]      # edit to split the work across Kaggle sessions
print(CFG)

## 2 — Data

In [ ]:
data, labels = sc.build_smap_dataset()
train_windows = {c: data[c]["normal_windows"] for c in sc.META_TRAIN}
pool = np.concatenate([train_windows[c] for c in sc.META_TRAIN]).astype(np.float32)
val_episodes = sc.fixed_episodes({c: data[c]["normal_windows"] for c in sc.META_VAL},
                                 CFG["val_episodes_per_task"], CFG["val_seed"],
                                 CFG["support_size"], CFG["query_size"])
test_windows = {c: sc.create_windows(data[c]["test"], sc.WINDOW, 1) for c in CFG["channels"]}
print(f"meta-train pool {pool.shape} | validation episodes {len(val_episodes)}")
for c in CFG["channels"]:
    print(f"  {c}: normal windows {len(data[c]['normal_windows'])}, test length {len(data[c]['test'])}, "
          f"anomalous fraction {data[c]['labels'].mean():.3f}")

## 3 — Finding earlier results and checkpoints (for resuming)

In [ ]:
def find_all(name):
    hits = [os.path.join(OUT, name)] if os.path.exists(os.path.join(OUT, name)) else []
    if os.path.isdir("/kaggle/input"):
        hits += [os.path.join(d, name) for d, _, f in os.walk("/kaggle/input") if name in f]
    return hits

def latest_checkpoint(name):
    best, best_step = None, -1
    for p in find_all(name):
        try:
            s = torch.load(p, map_location="cpu", weights_only=False).get("step", 0)
        except Exception:
            continue
        if s > best_step:
            best, best_step = p, s
    return best

def result_name(seed):
    return f"smap_corrected_seed{seed}{'_SMOKE' if SMOKE else ''}.json"

## 4 — Training for one seed

MAML, the static LSTM-AE and the MLP-AE all learn from the same 39 meta-training
channels. Finished models are saved and reused.

In [ ]:
def train_models(seed):
    models, info = {}, {}
    name = f"smap_maml_seed{seed}"
    done = find_all(f"{name}_best.pt")
    sc.seed_everything(seed)
    maml = sc.LSTMAutoencoder(25).to(DEVICE)
    if done:
        ck = torch.load(done[0], map_location=DEVICE, weights_only=False)
        maml.load_state_dict(ck["model_state_dict"]); info["maml"] = ck["info"]
        print("loaded", done[0])
    else:
        maml, info["maml"] = sc.train_maml(
            maml, train_windows, val_episodes, DEVICE, seed=seed, n_outer=CFG["n_outer"],
            val_every=CFG["val_every"], inner_lr=CFG["inner_lr"], inner_steps=CFG["inner_steps"],
            outer_lr=CFG["outer_lr"], tasks_per_batch=CFG["tasks_per_batch"],
            support_size=CFG["support_size"], query_size=CFG["query_size"],
            use_scheduler=CFG["use_scheduler"], ckpt_path=os.path.join(OUT, f"{name}_ckpt.pt"),
            resume_from=latest_checkpoint(f"{name}_ckpt.pt"))
        torch.save({"model_state_dict": maml.state_dict(), "info": info["maml"]},
                   os.path.join(OUT, f"{name}_best.pt"))
    models["maml"] = maml
    for key, cls in [("static", sc.LSTMAutoencoder), ("mlp", sc.MLPAutoencoder)]:
        fname = f"smap_{key}_seed{seed}.pt"
        sc.seed_everything(seed)
        m = cls(25).to(DEVICE)
        if find_all(fname):
            ck = torch.load(find_all(fname)[0], map_location=DEVICE, weights_only=False)
            m.load_state_dict(ck["model_state_dict"]); info[key] = ck["info"]
        else:
            m, info[key] = sc.train_conventional(m, pool, DEVICE, seed=seed,
                                                 max_epochs=CFG["static_max_epochs"],
                                                 patience=CFG["static_patience"])
            torch.save({"model_state_dict": m.state_dict(), "info": info[key]}, os.path.join(OUT, fname))
        models[key] = m
    return models, info

## 5 — Scoring for one seed

For every channel, shot count and support seed, the same K normal support windows (from
the channel's training file) are given to every method. Each adapted model scores the whole
test file point by point. The label-free threshold is the mean + 2 standard deviations of
the support-window errors (1.20 x the mean at 1 shot).

In [ ]:
def support_indices(ch, k, s):
    rng = np.random.RandomState([s, k, sc.META_TEST.index(ch)])
    return np.sort(rng.choice(len(data[ch]["normal_windows"]), k, replace=False))

def score_model(model, sup, ch, steps, lr):
    adapted = sc.inner_adapt(model, sc.to_tensor(sup, DEVICE), lr, steps) if steps else model
    errs = sc.window_errors(adapted, test_windows[ch], DEVICE)
    pw = sc.pointwise_from_window_errors(errs, len(data[ch]["test"]))
    tau = sc.label_free_threshold(sc.window_errors(adapted, sup, DEVICE))
    return sc.detection_metrics(pw, data[ch]["labels"], tau), adapted

def score_iforest(sup, ch, seed):
    if len(sup) == 1:
        return {"not_applicable": "one support window: Isolation Forest cannot be fitted meaningfully"}
    iso = IsolationForest(n_estimators=100, contamination="auto", random_state=seed).fit(sup.reshape(len(sup), -1))
    tw = test_windows[ch]
    errs = np.concatenate([-iso.score_samples(tw[i:i + 4096].reshape(len(tw[i:i + 4096]), -1))
                           for i in range(0, len(tw), 4096)])
    pw = sc.pointwise_from_window_errors(errs, len(data[ch]["test"]))
    tau = sc.label_free_threshold(-iso.score_samples(sup.reshape(len(sup), -1)))
    return sc.detection_metrics(pw, data[ch]["labels"], tau)

def old_f1_rule(y, s, tau):
    p = (s > tau).astype(int)
    return float(f1_score(y, p, zero_division=0)) if 0 < p.sum() < len(p) else 0.0

def evaluate(models, seed):
    res = {"pointwise": {}, "legacy_window_view": {}, "adaptation": {}}
    for ch in CFG["channels"]:
        for k in CFG["k_shots"]:
            for s in CFG["support_seeds"]:
                sup = data[ch]["normal_windows"][support_indices(ch, k, s)]
                key = f"{ch}|{k}|{s}"
                r = {}
                r["MAML-AE"], a = score_model(models["maml"], sup, ch, CFG["adapt_steps"], CFG["adapt_lr"])
                res["adaptation"][f"MAML-AE|{key}"] = sc.adaptation_report(models["maml"], a, sc.to_tensor(sup, DEVICE))
                r["MAML-AE (0 steps)"], _ = score_model(models["maml"], sup, ch, 0, CFG["adapt_lr"])
                r["Static-AE"], a = score_model(models["static"], sup, ch, CFG["adapt_steps"], CFG["adapt_lr"])
                res["adaptation"][f"Static-AE|{key}"] = sc.adaptation_report(models["static"], a, sc.to_tensor(sup, DEVICE))
                r["Static-AE (0 steps)"], _ = score_model(models["static"], sup, ch, 0, CFG["adapt_lr"])
                r["MLP-AE"], _ = score_model(models["mlp"], sup, ch, CFG["adapt_steps"], CFG["adapt_lr"])
                torch.manual_seed(seed * 1000 + s)
                r["MLP-scratch (legacy)"], _ = score_model(sc.MLPAutoencoder(25).to(DEVICE), sup, ch,
                                                           CFG["mlp_scratch_steps"], CFG["adapt_lr"])
                torch.manual_seed(seed * 1000 + s)
                r["LSTM-AE untrained (floor)"], _ = score_model(sc.LSTMAutoencoder(25).to(DEVICE), sup, ch, 0, 0.0)
                r["Isolation-Forest"] = score_iforest(sup, ch, s)
                res["pointwise"][key] = r
            for s in CFG["legacy_support_seeds"]:
                d = data[ch]
                sup, q, y, _, _ = sc.legacy_eval_query(d["normal_windows"], d["legacy_anomaly_windows"], k, s)
                r = {"query_anomaly_fraction": float(y.mean())}
                for name, base, steps in [("MAML-AE", models["maml"], CFG["legacy_adapt_steps"]),
                                          ("Static-AE", models["static"], CFG["legacy_adapt_steps"])]:
                    a = sc.inner_adapt(base, sc.to_tensor(sup, DEVICE), 0.01, steps)
                    e = sc.window_errors(a, q, DEVICE)
                    tau = sc.label_free_threshold(sc.window_errors(a, sup, DEVICE))
                    r[name] = {"roc_auc": float(roc_auc_score(y, e)), "f1_old_rule": old_f1_rule(y, e, tau)}
                res["legacy_window_view"][f"{ch}|{k}|{s}"] = r
        print(f"  seed {seed}: {ch} scored")
    return res

## 6 — Run all training seeds (finished seeds are skipped)

In [ ]:
for seed in TRAIN_SEEDS:
    prior = find_all(result_name(seed))
    if prior:
        print(f"seed {seed}: result found at {prior[0]}, skipping"); continue
    t0 = time.time()
    models, info = train_models(seed)
    res = evaluate(models, seed)
    sc.save_json(os.path.join(OUT, result_name(seed)),
                 {"experiment": "SMAP, corrected (Notebook_06)", "smoke_test": SMOKE,
                  "train_seed": seed, "config": CFG, "training": info, "results": res,
                  "device": str(DEVICE), "torch": torch.__version__,
                  "wall_seconds": time.time() - t0})
    print(f"seed {seed}: saved {result_name(seed)} ({time.time() - t0:.0f}s)")

## 7 — Summary across training seeds

For each method and shot count: the macro mean over the seven evaluation channels of the
mean over support draws, computed per training seed, then the mean, standard deviation and
95% confidence interval across training seeds. Paired differences use the training seed as
the unit, with two-sided t-tests and Wilcoxon tests. With fewer than two seeds only means
are shown. P-4 is summarised separately.

In [ ]:
runs = {seed: json.load(open(find_all(result_name(seed))[0])) for seed in CFG["train_seeds"]
        if find_all(result_name(seed))}
print("training seeds available:", sorted(runs))
METHODS = ["MAML-AE", "MAML-AE (0 steps)", "Static-AE", "Static-AE (0 steps)", "MLP-AE",
           "MLP-scratch (legacy)", "LSTM-AE untrained (floor)", "Isolation-Forest"]
METRICS = ["roc_auc", "pr_auc", "f1_at_tau", "oracle_best_f1", "trivial_all_positive_f1"]
eval_ch = [c for c in sc.EVAL_CHANNELS if c in CFG["channels"]]

def per_seed(run, method, k, metric, channels):
    vals = []
    for ch in channels:
        xs = [run["results"]["pointwise"][f"{ch}|{k}|{s}"][method].get(metric) for s in CFG["support_seeds"]]
        xs = [x for x in xs if x is not None]
        if xs: vals.append(np.mean(xs))
    return float(np.mean(vals)) if len(vals) == len(channels) else None

def describe(xs):
    xs = np.array([x for x in xs if x is not None], dtype=float)
    out = {"n_seeds": len(xs), "mean": float(xs.mean()) if len(xs) else None}
    if len(xs) > 1:
        sd = xs.std(ddof=1); h = stats.t.ppf(0.975, len(xs) - 1) * sd / np.sqrt(len(xs))
        out.update(sd=float(sd), ci95=[float(xs.mean() - h), float(xs.mean() + h)])
    return out

summary = {"pointwise_macro_eval_channels": {}, "pointwise_P-4": {}, "paired_differences_roc_auc": {},
           "legacy_view_macro_roc_auc": {}}
for k in CFG["k_shots"]:
    for m in METHODS:
        for met in METRICS:
            summary["pointwise_macro_eval_channels"][f"{m}|{k}|{met}"] = describe([per_seed(r, m, k, met, eval_ch) for r in runs.values()])
            if "P-4" in CFG["channels"]:
                summary["pointwise_P-4"][f"{m}|{k}|{met}"] = describe([per_seed(r, m, k, met, ["P-4"]) for r in runs.values()])
    for a, b in [("MAML-AE", "Static-AE"), ("MAML-AE", "MAML-AE (0 steps)"), ("Static-AE", "Static-AE (0 steps)"),
                 ("MAML-AE", "LSTM-AE untrained (floor)")]:
        d = [per_seed(r, a, k, "roc_auc", eval_ch) - per_seed(r, b, k, "roc_auc", eval_ch) for r in runs.values()]
        entry = describe(d)
        if len(d) > 1:
            entry["t_test_p_two_sided"] = float(stats.ttest_1samp(d, 0.0).pvalue)
            try: entry["wilcoxon_p_two_sided"] = float(stats.wilcoxon(d).pvalue)
            except ValueError: entry["wilcoxon_p_two_sided"] = None
        summary["paired_differences_roc_auc"][f"{a} minus {b}|{k}"] = entry
    for m in ["MAML-AE", "Static-AE"]:
        summary["legacy_view_macro_roc_auc"][f"{m}|{k}"] = describe([
            np.mean([np.mean([r["results"]["legacy_window_view"][f"{ch}|{k}|{s}"][m]["roc_auc"]
                              for s in CFG["legacy_support_seeds"]]) for ch in eval_ch]) for r in runs.values()])

def fmt(e):
    if e["mean"] is None: return "n/a"
    return f"{e['mean']:.3f}" + (f" [{e['ci95'][0]:.3f}, {e['ci95'][1]:.3f}]" if "ci95" in e else "")

for k in CFG["k_shots"]:
    print(f"\n{k}-shot, point-wise, macro over {len(eval_ch)} channels (mean [95% CI] over training seeds)")
    print(f"{'method':28s} {'ROC-AUC':>22s} {'PR-AUC':>22s} {'F1 at tau':>22s} {'oracle best-F1':>22s}")
    for m in METHODS:
        g = lambda met: fmt(summary["pointwise_macro_eval_channels"][f"{m}|{k}|{met}"])
        print(f"{m:28s} {g('roc_auc'):>22s} {g('pr_auc'):>22s} {g('f1_at_tau'):>22s} {g('oracle_best_f1'):>22s}")
    print(f"flag-everything F1 for reference: {fmt(summary['pointwise_macro_eval_channels'][f'MAML-AE|{k}|trivial_all_positive_f1'])}")
    for key, e in summary["paired_differences_roc_auc"].items():
        if key.endswith(f"|{k}"):
            print(f"  {key.split('|')[0]:45s} {fmt(e)}  p(t)={e.get('t_test_p_two_sided')}")
sc.save_json(os.path.join(OUT, f"smap_corrected_summary{'_SMOKE' if SMOKE else ''}.json"),
             {"smoke_test": SMOKE, "train_seeds_used": sorted(runs), "config": CFG, "summary": summary})

## 8 — How to read the results

- **MAML-AE minus Static-AE**: counts as a difference only if its 95% interval excludes 0.
- **MAML-AE minus MAML-AE (0 steps)**: if close to 0, adaptation does not change what the
  meta-learned model detects.
- **Untrained LSTM-AE (floor)**: a trained model that does not beat this floor has not
  shown that its training helps detection on these channels.
- **F1 at tau** is the only deployable F1. Compare it with the flag-everything F1, which
  depends only on the anomaly fraction. **Oracle best-F1** uses the labels to set the
  threshold and is an upper bound.
- **Legacy window view** exists only to line up with the old table. It is not a valid test.